# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۳: استراتژی‌های خرد کردن و تقسیم متن (Text Splitting Strategies)

### 🎂 تمثیل کیک عروسی: چرا خرد کردن متن یک هنر است؟
تصور کنید یک کیک عروسی بزرگ چندطبقه دارید و می‌خواهید آن را بین مهمان‌ها تقسیم کنید:
- **روش کورکورانه:** چشم‌ها را ببندید و کیک را به فواصل مساوی ۱۰ سانتی‌متری ببرید! نتیجه له شدن تزئینات، افتادن تکه‌های خامه خالص برای یک نفر و نان خالی برای دیگری است.
- **روش هوشمندانه:** نگاه کنید طبقات و مرزهای کیک کجا هستند و برش‌ها را در مرزهای طبیعی بزنید تا هر تکه یک برش زیبا، متوازن و کامل باشد.

در سیستم‌های RAG، متن‌های ما همان کیک هستند. اگر متن را بدون توجه به مرز کلمات و جملات برش دهیم (مثلاً دقیقاً در کاراکتر ۵۰۰ام)، ممکن است جمله‌ای از وسط نصف شود:
> چانک ۱: *«... پایتخت ایران شهر تهـ...»*
> چانک ۲: *«...ـران است.»*

مدل زبانی نه در چانک اول نام پایتخت را درمی‌یابد و نه در چانک دوم معنای کلمه «ـران» را! **معنا در مرز برش از بین رفت.**

> 💡 **تعریف مهندسی:** یک چانک (Chunk) صرفاً تکه‌ای از متن نیست؛ بلکه یک **«کپسول معنایی» (Semantic Capsule)** است.

### ۱. چرا باید متن‌ها را خرد (Chunk) کنیم؟

چرا کل کتاب یا مقاله را یکجا به مدل نمی‌دهیم؟
1. **محدودیت فیزیکی پنجره زمینه (Context Window):** مدل‌های زبانی ظرفیت حافظه کوتاه‌مدت محدودی دارند و ارسال متن‌های عظیم باعث هزینه سنگین مالی و خطای سرریز می‌شود.
2. **پدیده گم‌شدن در میانه (Lost in the Middle):** تحقیقات نشان داده مدل‌ها به ابتدا و انتهای ورودی دقت بیشتری دارند؛ اگر پاسخی در میانه یک متن ۱۰۰ صفحه‌ای باشد، مدل آن را نادیده گرفته یا گم می‌کند.

با تکنیک **Chunking**، ما اسناد را به قطعات کوچک و متوازن (مثلاً پاراگراف‌های ۵۰۰ تا ۱۰۰۰ کاراکتری) تقسیم می‌کنیم و در زمان پرسش، فقط ۳ یا ۴ چانک دقیقاً مرتبط را به مدل تحویل می‌دهیم.

### ۲. ساده‌ترین روش: `CharacterTextSplitter` (برش مکانیکی و ناآگاه از معنا)

کلاس `CharacterTextSplitter` همانند قصاب عمل می‌کند؛ به آن می‌گویید در فواصل ثابت (مثلاً هر ۵۰ یا ۱۰۰ کاراکتر) برش بزند.
بیایید خروجی این روش را روی یک متن فارسی ساخت‌یافته مشاهده کنیم:

In [1]:
from langchain_text_splitters import CharacterTextSplitter

long_text = """
مقدمه ای بر هوش مصنوعی
هوش مصنوعی (AI) شاخه ای از علوم رایانه است. این علم سعی دارد ماشین هایی بسازد که مثل انسان فکر کنند.
تاریخچه AI به دهه 1950 برمی گردد.

یادگیری ماشین
یکی از زیرشاخه های مهم AI، یادگیری ماشین (ML) است. در ML، ما به کامپیوتر صریحا برنامه نمی دهیم، بلکه اجازه می دهیم از داده ها یاد بگیرد.
"""

# تنظیم اسپلیتر ساده مکانیکی
simple_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=50,
    chunk_overlap=0
)

simple_chunks = simple_splitter.split_text(long_text.strip())

print(f"تعداد کل چانک‌ها: {len(simple_chunks)}\n")
for i, chunk in enumerate(simple_chunks, 1):
    print(f"--- چانک {i} ---")
    print(f"'{chunk}'")

تعداد کل چانک‌ها: 7

--- چانک 1 ---
'مقدمه ای بر هوش مصنوعی
هوش مصنوعی (AI) شاخه ای از'
--- چانک 2 ---
'علوم رایانه است. این علم سعی دارد ماشین هایی بسازد'
--- چانک 3 ---
'که مثل انسان فکر کنند.
تاریخچه AI به دهه 1950 برم'
--- چانک 4 ---
'ی گردد.

یادگیری ماشین
یکی از زیرشاخه های مهم AI،'
--- چانک 5 ---
'یادگیری ماشین (ML) است. در ML، ما به کامپیوتر صریح'
--- چانک 6 ---
'ا برنامه نمی دهیم، بلکه اجازه می دهیم از داده ها ی'
--- چانک 7 ---
'اد بگیرد.'


> ❌ **فاجعه در خروجی فوق را ببینید:**
> در چانک ۲ کلمه به صورت *«بساز»* بریده شده و در چانک ۳ با حرف تک‌افتاده *«د که مثل انسان فکر کنند...»* آغاز می‌شود!
> هیچ مدل هوشمندی نمی‌تواند متنی که کلماتش این‌گونه قطعه‌قطعه شده را به درستی تحلیل کند.

### ۳. استاندارد طلایی RAG: کلاس `RecursiveCharacterTextSplitter`

این کلاس ابزاری است که در **۹۹٪ پروژه‌های واقعی** از آن استفاده خواهید کرد.
الگوریتم بازگشتی (Recursive) دارای یک فهرست اولویت‌بندی‌شده از جداکننده‌ها است (به صورت پیش‌فرض: `["\n\n", "\n", " ", ""]`):

```
آیا می‌توان متن را از محل پاراگراف‌ها (\n\n) جدا کرد تا اندازه زیر chunk_size بماند؟
├── بله؟  --> بسیار عالی! برش در مرز پاراگراف انجام می‌شود.
└── خیر؟ --> پله دوم: برش از محل خطوط (\n).
            ├── موفق؟ --> پایان برش.
            └── ناموفق؟ --> پله سوم: برش از محل فاصله بین کلمات (" ") تا کلمات سالم بمانند.
                            └── در بدترین حالت (رشته طولانی بدون فاصله): برش کاراکتری ("").
```

بیایید همان متن را با `RecursiveCharacterTextSplitter` تقسیم کنیم:

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=["\n\n", "\n", " ", ""]
)

smart_chunks = recursive_splitter.create_documents([long_text.strip()])

print(f"تعداد چانک‌های هوشمند: {len(smart_chunks)}\n")
for i, chunk in enumerate(smart_chunks, 1):
    print(f"--- چانک {i} ---")
    print(f"'{chunk.page_content}'")

تعداد چانک‌های هوشمند: 7

--- چانک 1 ---
'مقدمه ای بر هوش مصنوعی'
--- چانک 2 ---
'هوش مصنوعی (AI) شاخه ای از علوم رایانه است. این علم سعی دارد ماشین هایی بسازد که مثل انسان فکر'
--- چانک 3 ---
'کنند.'
--- چانک 4 ---
'تاریخچه AI به دهه 1950 برمی گردد.'
--- چانک 5 ---
'یادگیری ماشین'
--- چانک 6 ---
'یکی از زیرشاخه های مهم AI، یادگیری ماشین (ML) است. در ML، ما به کامپیوتر صریحا برنامه نمی دهیم،'
--- چانک 7 ---
'بلکه اجازه می دهیم از داده ها یاد بگیرد.'


### ۴. شخصی‌سازی و بومی‌سازی برای زبان فارسی (Persian Optimization)

زبان فارسی دارای ویژگی‌های نگارشی منحصربه‌فردی است:
- استفاده فراوان از **نیم‌فاصله (`\u200c`)** (مانند «می‌شود»، «برنامه‌نویسی»).
- استفاده از علائم نگارشی فارسی مثل **ویرگول فارسی (`،`)** و **علامت سوال (`؟`)**.

با اضافه کردن این نشانه‌ها به لیست `separators`، به اسپلیتر می‌فهمانیم که پایان جمله یا ویرگول مکان‌های بسیار مناسبی برای برش زدن هستند، حتی اگر سطر هنوز تمام نشده باشد:

In [3]:
# اسپلیتر بهینه‌سازی‌شده برای رسم‌الخط و جمله‌بندی زبان فارسی
persian_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=0,
    separators=[
        "\n\n",   # اولویت ۱: پاراگراف
        "\n",     # اولویت ۲: سطر جدید
        ". ",     # اولویت ۳: پایان جمله با نقطه
        "؟ ",     # اولویت ۴: علامت سوال
        "! ",     # اولویت ۵: علامت تعجب
        "، ",     # اولویت ۶: ویرگول فارسی
        " ",      # اولویت ۷: فاصله بین کلمات
        "\u200c",  # اولویت ۸: نیم‌فاصله
        ""        # اولویت آخر: کاراکتر منفرد (در صورت لزوم)
    ]
)

persian_sample = "مدل‌های زبانی بزرگ با مکانیزم توجه آموزش می‌بینند، اما برای پاسخگویی به اسناد اختصاصی سازمان‌ها به سیستم‌های RAG نیاز دارند. این سیستم‌ها متن‌ها را بازیابی کرده و پاسخ‌های دقیق ارائه می‌دهند."

persian_chunks = persian_splitter.split_text(persian_sample)

print(f"تعداد چانک‌های فارسی: {len(persian_chunks)}\n")
for i, c in enumerate(persian_chunks, 1):
    print(f"چانک {i}: '{c}'")

تعداد چانک‌های فارسی: 3

چانک 1: 'مدل‌های زبانی بزرگ با مکانیزم توجه آموزش می‌بینند'
چانک 2: '، اما برای پاسخگویی به اسناد اختصاصی سازمان‌ها به سیستم‌های RAG نیاز دارند'
چانک 3: '. این سیستم‌ها متن‌ها را بازیابی کرده و پاسخ‌های دقیق ارائه می‌دهند.'


### ۵. راهنمای انتخاب اندازه چانک (`chunk_size`): کوچک یا بزرگ؟

| اندازه چانک | مزایا | معایب و چالش‌ها | موارد کاربرد پیشنهادی |
| :--- | :--- | :--- | :--- |
| **کوچک (۱۰۰ تا ۴۰۰ کاراکتر)** | جستجوی فوق‌العاده متمرکز و دقیق بر روی یک جمله یا فکت منفرد. | **کمبود کانتکست (Context Loss)**؛ ابهام در مراجع ضمایر (مثلاً: *«او موافقت کرد»* بدون اینکه مشخص باشد *او* کیست). | سوالات تک‌فکتی و دایره‌المعارفی، واژه‌نامه‌ها، FAQهای کوتاه. |
| **متوسط (۵۰۰ تا ۱۰۰۰ کاراکتر)** | تعادل عالی میان حفظ کانتکست و دقت بازیابی؛ محبوب‌ترین انتخاب. | نیاز به تنظیم دقیق همپوشانی (Overlap). | اکثر مقالات، کاتالوگ‌ها، اسناد اداری و سناریوهای عمومی RAG. |
| **بزرگ (۱۰۰۰ تا ۲۰۰۰+ کاراکتر)** | حفظ پیوستگی کامل ایده‌ها، استدلال‌ها و مثال‌های طولانی. | افزایش **نویز** در ورودی مدل و افت رزولوشن جستجوی معنایی. | متون حقوقی، قراردادها، تحلیل‌های پیچیده فلسفی و گزارش‌های مالی. |

### 🎯 جمع‌بندی نکات کلیدی این بخش:

1. **تعریف چانک:** یک «کپسول معنایی» که باید یک گزاره یا ایده کامل را بدون قطع ناگهانی کلمات منتقل کند.
2. **شکست CharacterTextSplitter:** برش‌های کورکورانه باعث نصف شدن کلمات و از دست رفتن معنا می‌شوند.
3. **قدرت RecursiveCharacterTextSplitter:** برش سلسله‌مراتبی از پاراگراف تا کلمه، حفظ سلامت کلمات و انسجام معنایی را تضمین می‌کند.
4. **بومی‌سازی فارسی:** با افزودن نشانه‌های `.`، `،`، `؟` و نیم‌فاصله `\u200c` به جداکننده‌ها، متن فارسی با دقت خیره‌کننده خرد می‌شود.

**مسئله جلسه بعد:**
حتی در بهترین اسپلیترها، اگر یک جمله در انتهای چانک اول بیفتد، ممکن است ادامه‌اش در ابتدای چانک دوم باشد و فاعل از فعل جدا شود.
چگونه با تکنیک **همپوشانی (Chunk Overlap)** و **حفظ متادیتا (Metadata Preservation)** ارتباط دو تکه مجاور را بیمه کنیم؟ در درس بعدی خواهیم دید.